In [2]:
#!pip install scikit-learn pandas seaborn joblib google-cloud-storage fastapi uvicorn pyngrok nest_asyncio

In [4]:
#!pip install google-cloud-bigquery google-cloud-storage \
             pyarrow db-dtypes \
             scikit-learn pandas joblib \
             fastapi uvicorn pyngrok nest_asyncio

IndentationError: unexpected indent (2409901554.py, line 2)

In [6]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "turing-alcove-384403"

client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT
    species,
    culmen_length_mm,
    culmen_depth_mm,
    flipper_length_mm,
    body_mass_g
FROM
    `bigquery-public-data.ml_datasets.penguins`
WHERE
    species IS NOT NULL
"""

df = client.query(query).to_dataframe()

print(df.head())
print(df.shape)

                                     species  culmen_length_mm  \
0        Adelie Penguin (Pygoscelis adeliae)              36.6   
1        Adelie Penguin (Pygoscelis adeliae)              39.8   
2        Adelie Penguin (Pygoscelis adeliae)              40.9   
3  Chinstrap penguin (Pygoscelis antarctica)              46.5   
4        Adelie Penguin (Pygoscelis adeliae)              37.3   

   culmen_depth_mm  flipper_length_mm  body_mass_g  
0             18.4              184.0       3475.0  
1             19.1              184.0       4650.0  
2             18.9              184.0       3900.0  
3             17.9              192.0       3500.0  
4             16.8              192.0       3000.0  
(344, 5)


In [8]:
df = df.dropna()

X = df[[
    'culmen_length_mm',
    'culmen_depth_mm',
    'flipper_length_mm',
    'body_mass_g'
]]

y = df['species']

print(X.head())
print(y.head())

   culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g
0              36.6             18.4              184.0       3475.0
1              39.8             19.1              184.0       4650.0
2              40.9             18.9              184.0       3900.0
3              46.5             17.9              192.0       3500.0
4              37.3             16.8              192.0       3000.0
0          Adelie Penguin (Pygoscelis adeliae)
1          Adelie Penguin (Pygoscelis adeliae)
2          Adelie Penguin (Pygoscelis adeliae)
3    Chinstrap penguin (Pygoscelis antarctica)
4          Adelie Penguin (Pygoscelis adeliae)
Name: species, dtype: object


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

Accuracy: 0.9420289855072463


In [12]:
import joblib

MODEL_FILE = "penguin_model.joblib"

joblib.dump(model, MODEL_FILE)

print("Model saved successfully!")
print(model,MODEL_FILE,"Model saved successfully!")


Model saved successfully!
RandomForestClassifier(random_state=42) penguin_model.joblib Model saved successfully!


In [13]:
from google.cloud import storage

PROJECT_ID = "turing-alcove-384403"
BUCKET_NAME = "my2026model"

storage_client = storage.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)

blob = bucket.blob("models/v1/penguin_model.joblib")

blob.upload_from_filename(MODEL_FILE)

print("Uploaded to GCS successfully!")

Uploaded to GCS successfully!


In [14]:
import requests

url = "http://localhost:8000/predict"

data = {
    "culmen_length_mm": 39.1,
    "culmen_depth_mm": 18.7,
    "flipper_length_mm": 181,
    "body_mass_g": 3750
}

response = requests.post(url, params=data)

print(response.json())

{'prediction': 'Adelie Penguin (Pygoscelis adeliae)'}


In [ ]:
#uvicorn app:app --host 0.0.0.0 --port 8000